# Turning a text into tokens

In [ ]:
import requests

url = "https://raw.githubusercontent.com/lucasaraujomedeiros/llm-lab/main/frankenstein.txt"
file_path = "frankenstein.txt"

response = requests.get(url, timeout=30)
response.raise_for_status()

with open(file_path, "wb") as f:
    f.write(response.content)

In [ ]:
with open("frankenstein.txt", encoding="utf-8") as f:
    raw_text = f.read()


- Breaking the text into smaller token


In [ ]:
import re
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item for item in preprocessed if item.strip()]
len(preprocessed)

85633

# Creating a Vocabulary

In [ ]:
all_words = sorted(set(preprocessed))
len(all_words)

7463

In [ ]:
all_words[:100]

['!',
 '"',
 "'",
 '(',
 ')',
 ',',
 '--',
 '.',
 '1',
 '10',
 '11',
 '11th',
 '12',
 '12th',
 '13',
 '13th',
 '14',
 '15',
 '16',
 '17',
 '18',
 '18th',
 '19',
 '2',
 '20',
 '21',
 '22',
 '23',
 '24',
 '26th',
 '27th',
 '28th',
 '2nd',
 '3',
 '31st',
 '4',
 '5',
 '5th',
 '6',
 '7',
 '7th',
 '8',
 '9',
 '9th',
 ':',
 ';',
 '?',
 'A',
 'Abbey',
 'Abhorred',
 'About',
 'Absence',
 'Accordingly',
 'Accursed',
 'Adam',
 'Adieu',
 'Africa',
 'After',
 'Again',
 'Agatha',
 'Agrippa',
 'Ah',
 'Alas',
 'Albertus',
 'All',
 'Almost',
 'Alphonse',
 'Alpine',
 'Alps',
 'Although',
 'Am',
 'America',
 'American',
 'Amidst',
 'Among',
 'An',
 'Ancient',
 'And',
 'Andes',
 'Andrew',
 'Angel',
 'Angelica',
 'Anguish',
 'Another',
 'Answer',
 'Arab',
 'Arabian',
 'Arabic',
 'Archangel',
 'Are',
 'Ariosto',
 'Armada',
 'Arthur',
 'Arve',
 'Arveiron',
 'As',
 'Asia',
 'Asiatics',
 'At',
 'Atlantic']

- Enumerating the tokens and making a vocab

In [ ]:
vocab = {s:i for i,s in enumerate(all_words)}
for i, item in enumerate(vocab.items()):
  print(item)
  if i >= 99:
    break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
('1', 8)
('10', 9)
('11', 10)
('11th', 11)
('12', 12)
('12th', 13)
('13', 14)
('13th', 15)
('14', 16)
('15', 17)
('16', 18)
('17', 19)
('18', 20)
('18th', 21)
('19', 22)
('2', 23)
('20', 24)
('21', 25)
('22', 26)
('23', 27)
('24', 28)
('26th', 29)
('27th', 30)
('28th', 31)
('2nd', 32)
('3', 33)
('31st', 34)
('4', 35)
('5', 36)
('5th', 37)
('6', 38)
('7', 39)
('7th', 40)
('8', 41)
('9', 42)
('9th', 43)
(':', 44)
(';', 45)
('?', 46)
('A', 47)
('Abbey', 48)
('Abhorred', 49)
('About', 50)
('Absence', 51)
('Accordingly', 52)
('Accursed', 53)
('Adam', 54)
('Adieu', 55)
('Africa', 56)
('After', 57)
('Again', 58)
('Agatha', 59)
('Agrippa', 60)
('Ah', 61)
('Alas', 62)
('Albertus', 63)
('All', 64)
('Almost', 65)
('Alphonse', 66)
('Alpine', 67)
('Alps', 68)
('Although', 69)
('Am', 70)
('America', 71)
('American', 72)
('Amidst', 73)
('Among', 74)
('An', 75)
('Ancient', 76)
('And', 77)
('Andes', 78)
('Andrew', 79)
('Angel', 80

- Adding special tokens

In [ ]:
all_words.extend(["[EOF]", "[UNK]"])
vocab = {s:i for i,s in enumerate(all_words)}
len(vocab)

7465

In [ ]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

('youth', 7460)
('youthful', 7461)
('zeal', 7462)
('[EOF]', 7463)
('[UNK]', 7464)


# Creating a tokenizer

In [ ]:
class myTokenizer:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}

    def encode(self, text):
        text_preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        text_preprocessed = [item.strip()  for item in text_preprocessed if item.strip()]
        text_preprocessed = [item if item in self.str_to_int else "[UNK]" for item in text_preprocessed]

        idx = [self.str_to_int[s] for s in text_preprocessed]
        return idx

    def decode(self, idx):
        text = " ".join([self.int_to_str[i] for i in idx])
        text = re.sub(r'\s+([;,.?!"()\'])', r'\1', text)
        return text

In [ ]:
tokenizer = myTokenizer(vocab)
text = "Hello, do you like tea? [EOF] In the sunlit terraces of the palace."

tokenizer.encode(text)

[7464,
 5,
 2491,
 7451,
 4276,
 7464,
 46,
 7463,
 307,
 6682,
 7464,
 7464,
 4801,
 6682,
 4932,
 7]

In [ ]:
tokenizer.decode(tokenizer.encode(text))

'[UNK], do you like [UNK]? [EOF] In the [UNK] [UNK] of the palace.'

# Byte Pair Encoding

In [ ]:
import importlib
import tiktoken

In [ ]:
text = "Hello, do you like tea? [EOF] In the sunlit terraces of the someunknownpalace."

tokenizer = tiktoken.get_encoding("gpt2")
tokenizer.encode(text, allowed_special={"[EOF]"})

[15496,
 11,
 466,
 345,
 588,
 8887,
 30,
 685,
 4720,
 37,
 60,
 554,
 262,
 4252,
 18250,
 8812,
 2114,
 286,
 262,
 617,
 34680,
 18596,
 558,
 13]

In [ ]:
tokenizer.decode(tokenizer.encode(text, allowed_special={"[EOF]"}))

'Hello, do you like tea? [EOF] In the sunlit terraces of the someunknownpalace.'

In [ ]:
enc_text = tokenizer.encode(raw_text, allowed_special={"[EOF]"})
print(len(enc_text))

102697


# Creating a Dataset

In [ ]:
import torch

In [ ]:
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt, allowed_special={"[EOF]"})
        assert len(token_ids) > max_length, "Number of tokenized inputs must at least be equal to max_length+1"

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [ ]:
def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [ ]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[17439, 37975,    11,   198],
        [  198,   273,   262, 12495],
        [42696,   628,   198,  1525],
        [  198,   198, 24119,   370],
        [  692,  6440,  3323,   357],
        [13482,  5404,     8, 46854],
        [  628,   628,   198, 45708],
        [  352,   628,   198,  1273]])

Targets:
 tensor([[37975,    11,   198,   198],
        [  273,   262, 12495, 42696],
        [  628,   198,  1525,   198],
        [  198, 24119,   370,   692],
        [ 6440,  3323,   357, 13482],
        [ 5404,     8, 46854,   628],
        [  628,   198, 45708,   352],
        [  628,   198,  1273,    13]])
